In [1]:
import os

def print_tree(start_path="."):
    for root, dirs, files in os.walk(start_path):
        if "venv" in root:
            continue
        level = root.replace(start_path, "").count(os.sep)
        indent = " " * 4 * level
        print(f"{indent}{os.path.basename(root)}/")
        subindent = " " * (4 * (level + 1))
        for f in files:
            print(f"{subindent}{f}")

print_tree()

./
    config.py
    requirements.txt
    repo_overview.ipynb
    test_routes.py
    README.md
    app.py
    test_output.txt
    init_db.py
    anaconda_projects/
        db/
            project_filebrowser.db
    uploads/
    utils/
        files.py
        __init__.py
        __pycache__/
            files.cpython-313.pyc
            files.cpython-312.pyc
            __init__.cpython-312.pyc
    models/
        user.py
        __init__.py
        note.py
        __pycache__/
            user.cpython-312.pyc
            user.cpython-313.pyc
            note.cpython-312.pyc
            note.cpython-313.pyc
            __init__.cpython-312.pyc
    templates/
        note_detail.html
    .ipynb_checkpoints/
        repo_overview-checkpoint.ipynb
    routes/
        auth.py
        __init__.py
        export.py
        notes.py
        admin.py
        __pycache__/
            notes.cpython-313.pyc
            notes.cpython-312.pyc
            auth.cpython-313.pyc
            auth.cpytho

In [2]:
py_files = []
total_lines = 0

for root, dirs, files in os.walk("."):
    if "venv" in root:
        continue
    for file in files:
        if file.endswith(".py"):
            path = os.path.join(root, file)
            py_files.append(path)
            with open(path, "r", encoding="utf-8", errors="ignore") as f:
                total_lines += sum(1 for _ in f)

print("Python files:", len(py_files))
print("Total lines:", total_lines)

for f in py_files:
    print(f)

Python files: 14
Total lines: 955
./config.py
./test_routes.py
./app.py
./init_db.py
./utils/files.py
./utils/__init__.py
./models/user.py
./models/__init__.py
./models/note.py
./routes/auth.py
./routes/__init__.py
./routes/export.py
./routes/notes.py
./routes/admin.py


## Project Overview

SecureNote is a Flask-based REST API that allows users to register, authenticate, create and search notes, and export data. The system also includes admin functionality and file handling features.

## Key Components

- app.py: main application entry point
- routes/: handles API endpoints (auth, notes, admin, export)
- models/: database structure and data handling
- utils/: helper functions such as file handling
- templates/: HTML views
- uploads/: stored uploaded files

## Attack Surface

The main attack surface includes endpoints that accept user input. This includes authentication routes, note creation and search endpoints, export functionality, and file download routes.

Key exposed endpoints include:
- /register, /login
- /notes
- /notes/search
- /export
- /files/<filename>
- /admin/...

These areas are most likely to contain vulnerabilities such as injection attacks, improper input validation, and insecure file handling.

## utils/files.py

In [5]:
with open("utils/files.py", "r") as f:
    print(f.read())

"""
SecureNote – File Utilities
Helper functions for serving uploaded or generated files.
"""

import os
from flask import send_file, abort

from config import Config


def download_file(filename: str):
    """Serve a file from the uploads directory.

    Parameters
    ----------
    filename : str
        Name (or relative path) of the requested file.
    """
    filepath = os.path.join(Config.UPLOAD_FOLDER, filename)

    if not os.path.isfile(filepath):
        abort(404, description="File not found")

    return send_file(filepath, as_attachment=True)



- `utils/files.py`: Handles file download functionality
- This is a high-risk area because it constructs file paths using user-controlled input without validation
- This may introduce a path traversal vulnerability, allowing attackers to access files outside the intended directory

## routes/auth.py

In [6]:
with open("routes/auth.py", "r") as f:
    print(f.read())

"""
SecureNote – Authentication Routes
Handles user registration, login, and logout.
"""

import logging
from functools import wraps

from flask import (
    Blueprint,
    request,
    jsonify,
    session,
    redirect,
    url_for,
)

from models.user import create_user, get_user_by_username, verify_password

logger = logging.getLogger(__name__)

auth_bp = Blueprint("auth", __name__)


# ── Helpers ─────────────────────────────────────────────────────────────────

def login_required(f):
    """Decorator that rejects unauthenticated requests."""
    @wraps(f)
    def wrapper(*args, **kwargs):
        if "user_id" not in session:
            return jsonify({"error": "Authentication required"}), 401
        return f(*args, **kwargs)
    return wrapper


def admin_required(f):
    """Decorator that restricts access to admin users."""
    @wraps(f)
    @login_required
    def wrapper(*args, **kwargs):
        if session.get("role") != "admin":
            return jsonify({"error": "Admin 

- `routes/auth.py`: Handles user registration, login, and session management
- This is a high-risk area because it processes authentication data
- The login route logs raw passwords during login attempts, which may expose sensitive credentials if logs are accessed or leaked

## routes/notes.py

In [10]:
with open("routes/notes.py", "r") as f:
    print(f.read())

"""
SecureNote – Notes Routes
CRUD endpoints and search for user notes.
"""

import logging

from flask import Blueprint, request, jsonify, session, render_template

from models.note import (
    create_note,
    get_note,
    get_notes_for_user,
    update_note,
    delete_note,
    search_notes,
)
from routes.auth import login_required

logger = logging.getLogger(__name__)

notes_bp = Blueprint("notes", __name__)


# ── List & Search ──────────────────────────────────────────────────────────

@notes_bp.route("/notes", methods=["GET"])
@login_required
def list_notes():
    """Return all notes for the logged-in user."""
    user_id = session["user_id"]
    notes = get_notes_for_user(user_id)
    return jsonify([dict(n) for n in notes])


@notes_bp.route("/notes/search", methods=["GET"])
@login_required
def search():
    """Full-text search across note titles for the current user."""
    user_id = session["user_id"]
    query = request.args.get("q", "")
    if not query:
        return 

- `routes/notes.py`: Handles note creation, search, viewing, updating, and deletion
- This is a high-risk area because it processes user-generated content and renders note data into HTML
- Rendering stored note content in a template may introduce cross-site scripting (XSS) risk if output is not properly escaped

## routes/export.py

In [13]:
with open("routes/export.py", "r") as f:
    print(f.read())

"""
SecureNote – Export / Import Routes
Endpoints for exporting notes to various formats and importing note
collections from serialized payloads.
"""

import base64
import logging
import os
import pickle
import tempfile

from flask import Blueprint, request, jsonify, session, send_file

from models.note import get_notes_for_user, get_note
from routes.auth import login_required

logger = logging.getLogger(__name__)

export_bp = Blueprint("export", __name__, url_prefix="/export")


@export_bp.route("/csv", methods=["GET"])
@login_required
def export_csv():
    """Export all of the current user's notes as a CSV file."""
    import csv
    import io

    notes = get_notes_for_user(session["user_id"])

    buf = io.StringIO()
    writer = csv.writer(buf)
    writer.writerow(["id", "title", "content", "created_at"])
    for n in notes:
        writer.writerow([n["id"], n["title"], n["content"], n["created_at"]])

    output = io.BytesIO(buf.getvalue().encode("utf-8"))
    output.seek(0)
    

- `routes/export.py`: Handles note export and import functionality
- This is a high-risk area because it processes serialized data, builds shell commands, and inserts user-controlled content into generated files
- The import route uses unsafe pickle deserialization, and the PDF export route may introduce command injection risk through unsanitized filename input

## routes/admin.py

In [14]:
with open("routes/admin.py", "r") as f:
    print(f.read())

"""
SecureNote – Admin Routes
Endpoints for user management (listing, deleting).
These are intended for administrator use only.
"""

import logging

from flask import Blueprint, jsonify, request

from models.user import list_all_users, delete_user, get_user_by_id

logger = logging.getLogger(__name__)

admin_bp = Blueprint("admin", __name__, url_prefix="/admin")


@admin_bp.route("/users", methods=["GET"])
def get_all_users():
    """List every registered user (id, username, role)."""
    users = list_all_users()
    return jsonify([dict(u) for u in users])


@admin_bp.route("/users/<int:user_id>", methods=["GET"])
def get_user(user_id):
    """Retrieve a single user's details."""
    user = get_user_by_id(user_id)
    if user is None:
        return jsonify({"error": "User not found"}), 404
    return jsonify({
        "id": user["id"],
        "username": user["username"],
        "role": user["role"],
    })


@admin_bp.route("/users/<int:user_id>", methods=["DELETE"])
def remove_use

- `routes/admin.py`: Handles administrative user management functions such as listing users, deleting users, and changing roles
- This is a high-risk area because it exposes sensitive administrative actions
- The admin routes do not enforce authentication or admin authorization, which may allow unauthorized users to access or modify user accounts

## models/user.py

In [15]:
with open("models/user.py", "r") as f:
    print(f.read())

"""
SecureNote – User Model
Handles user creation, password storage, and authentication lookups.
"""

import hashlib
import sqlite3

from config import Config


def get_db():
    """Return a connection to the SQLite database."""
    conn = sqlite3.connect(Config.DATABASE_PATH)
    conn.row_factory = sqlite3.Row
    return conn


# ── Password helpers ────────────────────────────────────────────────────────

def hash_password(password: str) -> str:
    """Return a hex-digest hash of *password* for storage."""
    return hashlib.md5(password.encode()).hexdigest()


def verify_password(password: str, stored_hash: str) -> bool:
    """Check a plain-text *password* against a *stored_hash*."""
    return hash_password(password) == stored_hash


# ── CRUD ────────────────────────────────────────────────────────────────────

def create_user(username: str, password: str, role: str = "user") -> int:
    """Insert a new user and return the new row id."""
    db = get_db()
    cursor = db.execute(

- `models/user.py`: Handles user creation, password storage, and authentication lookups
- This is a high-risk area because it manages credential security
- Passwords are hashed using MD5, which is an outdated and insecure hashing algorithm that may make stored credentials easier to crack if the database is compromised

## config.py

In [16]:
with open("config.py", "r") as f:
    print(f.read())

"""
SecureNote – Application Configuration
Centralized configuration for the SecureNote Flask application.
"""

import os


class Config:
    """Base configuration."""

    # ── Security ────────────────────────────────────────────────────────────
    SECRET_KEY = "supersecretkey123"                    # used for session signing
    SESSION_COOKIE_HTTPONLY = True

    # ── Database ────────────────────────────────────────────────────────────
    DATABASE_PATH = os.path.join(
        os.path.dirname(os.path.abspath(__file__)), "securenote.db"
    )

    # ── Upload / Export ─────────────────────────────────────────────────────
    UPLOAD_FOLDER = os.path.join(
        os.path.dirname(os.path.abspath(__file__)), "uploads"
    )
    MAX_CONTENT_LENGTH = 16 * 1024 * 1024  # 16 MB

    # ── Logging ─────────────────────────────────────────────────────────────
    LOG_LEVEL = "DEBUG"
    LOG_FILE = "securenote.log"


class DevelopmentConfig(Config):
    """Development-specific settings."""



- `config.py`: Handles application-wide security, database, upload, and logging settings
- This is a high-risk area because insecure configuration can expose the entire application
- The file contains a hardcoded secret key and enables debug mode in the production configuration, which may weaken session security and expose sensitive internal details

## Priority Files for Analysis

Based on the repository review, the following files are the highest priority for vulnerability analysis:

1. `routes/export.py` – unsafe deserialization and command execution
2. `routes/admin.py` – missing authentication and authorization
3. `utils/files.py` – potential path traversal
4. `models/user.py` – weak password hashing
5. `routes/auth.py` – credential exposure in logs

These files represent the most critical areas of the application where high-severity vulnerabilities are likely to exist.